In [4]:
import json
import time
from pathlib import Path

import pandas as pd
import requests

BASE_URL = "https://clinicaltrials.gov/api/v2/studies"

def fetch_all_studies(condition, page_size=1000, max_page=50):
    """Download every study matching a condition, and download one page at a time"""
    params = {"query.cond": condition, "pageSize": page_size}
    studies = []
    for page in range(max_page):
        response = requests.get(BASE_URL, timeout=60, params=params)
        response.raise_for_status()
        data = response.json()
        studies.extend(data.get("studies",[]))
        print(f"Page {page+1}: {len(studies)} studies so far")
        token = data.get("nextPageToken")
        if token is None:
            break
        params["pageToken"] = token
        time.sleep(1)
    return studies

In [19]:
studies = fetch_all_studies("non-small cell lung cancer") #Run once
print(len(studies))

Page 1: 1000 studies so far
Page 2: 2000 studies so far
Page 3: 3000 studies so far
Page 4: 4000 studies so far
Page 5: 5000 studies so far
Page 6: 6000 studies so far
Page 7: 7000 studies so far
Page 8: 8000 studies so far
Page 9: 8636 studies so far
8636


In [20]:
Path("../data/raw").mkdir(parents=True, exist_ok=True)

with open("../data/raw/nsclc_studies.json", "w") as f:
    json.dump(studies, f)

In [7]:
with open("../data/raw/nsclc_studies.json") as f:
    studies = json.load(f)
print(len(studies))

8636


In [6]:
def flatten_study(study):
    ps = study.get("protocolSection", {})
    ident = ps.get("identificationModule", {})
    status = ps.get("statusModule", {})
    design = ps.get("designModule", {})
    sponsor = ps.get("sponsorCollaboratorsModule", {}).get("leadSponsor", {})
    return {
        "nct_id": ident.get("nctId"),
        "title": ident.get("briefTitle"),
        "status": status.get("overallStatus"),
        "start_date": status.get("startDateStruct", {}).get("date"),
        "why_stopped": status.get("whyStopped"),
        "study_type": design.get("studyType"),
        "phases": ",".join(design.get("phases", [])),
        "enrollment": design.get("enrollmentInfo", {}).get("count"),
        "sponsor": sponsor.get("name"),
        "sponsor_class": sponsor.get("class"),
        "has_results": study.get("hasResults"),}
    

In [21]:
df = pd.DataFrame([flatten_study(s) for s in studies])
df.to_csv("../data/processed/nsclc_trials_raw.csv", index=False)
print(df.shape)
df.head()

(8636, 11)


,nct_id,title,status,start_date,why_stopped,study_type,phases,enrollment,sponsor,sponsor_class,has_results
0,NCT00266877,Study Evaluating the Safety Of HKI-272 (Nerati...,COMPLETED,2005-12,NaN,INTERVENTIONAL,PHASE2,172.0,"Puma Biotechnology, Inc.",INDUSTRY,True
1,NCT04207775,Real World Study in Locally Advanced or Metast...,COMPLETED,2020-03-30,NaN,OBSERVATIONAL,,300.0,AstraZeneca,INDUSTRY,False
2,NCT00126581,Erlotinib Hydrochloride With or Without Carbop...,COMPLETED,2005-08-15,NaN,INTERVENTIONAL,PHASE2,188.0,National Cancer Institute (NCI),NIH,True
3,NCT05764928,Evaluate the Safety and Efficacy of BLEX 404 O...,NOT_YET_RECRUITING,2026-12-01,NaN,INTERVENTIONAL,"PHASE1,PHASE2",32.0,Rgene Corporation,INDUSTRY,False
4,NCT04450732,Safety of GQ1001 in Adult Patients With HER2-P...,UNKNOWN,2020-07-07,NaN,INTERVENTIONAL,PHASE1,96.0,"GeneQuantum Healthcare (Suzhou) Co., Ltd.",INDUSTRY,False


In [10]:
df.info()                         # columns, data types, non-missing counts
df.isna().sum()                   # missing values per column
df["study_type"].value_counts()   # how many interventional vs observational
df["phases"].value_counts()       # every phase combination that exists
df["status"].value_counts()
df["sponsor_class"].value_counts()
df["nct_id"].is_unique

<class 'pandas.DataFrame'>
RangeIndex: 8636 entries, 0 to 8635
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   nct_id         8636 non-null   str    
 1   title          8636 non-null   str    
 2   status         8636 non-null   str    
 3   start_date     8552 non-null   str    
 4   why_stopped    1271 non-null   str    
 5   study_type     8636 non-null   str    
 6   phases         7249 non-null   str    
 7   enrollment     8437 non-null   float64
 8   sponsor        8636 non-null   str    
 9   sponsor_class  8636 non-null   str    
 10  has_results    8636 non-null   bool   
dtypes: bool(1), float64(1), str(9)
memory usage: 683.2 KB


True

In [ ]:
# keep_default_na=False stops pandas reading the phase label "NA" as missing data
df = pd.read_csv("../data/processed/nsclc_trials_raw.csv",
                 keep_default_na=False, na_values=[""])

trials = df[df["study_type"] == "INTERVENTIONAL"].copy()

# Dates come as "2019-05" or "2019-05-14"; the first 4 characters are the year
trials["start_year"] = pd.to_numeric(trials["start_date"].str[:4],
                                     errors="coerce").astype("Int64")

phase_labels = {
    "EARLY_PHASE1": "Early Phase 1",
    "PHASE1": "Phase 1",
    "PHASE1,PHASE2": "Phase 1/2",
    "PHASE2": "Phase 2",
    "PHASE2,PHASE3": "Phase 2/3",
    "PHASE3": "Phase 3",
    "PHASE4": "Phase 4",
    "NA": "Not applicable",
}
trials["phase"] = trials["phases"].map(phase_labels).fillna("Not specified")

sponsor_labels = {
    "INDUSTRY": "Industry",
    "NIH": "NIH",
    "OTHER": "Academic or hospital",
    "OTHER_GOV": "Other government",
    "FED": "Other government",
}
trials["sponsor_type"] = trials["sponsor_class"].map(sponsor_labels).fillna("Other")

trials.to_csv("../data/processed/nsclc_trials_clean.csv", index=False)

In [16]:
# Spot check: compare 5 random cleaned rows against the live API
def same(mine, live):
    """True if two values match, treating blanks/missing as equal and 172.0 == 172."""
    mine_missing = pd.isna(mine) or mine == ""
    live_missing = live is None or live == ""
    if mine_missing or live_missing:
        return mine_missing and live_missing
    try:
        return float(mine) == float(live)
    except (TypeError, ValueError):
        return str(mine) == str(live)

sample = trials.sample(5, random_state=1)

for _, row in sample.iterrows():
    nct = row["nct_id"]
    response = requests.get(f"https://clinicaltrials.gov/api/v2/studies/{nct}", timeout=30)
    response.raise_for_status()
    live = flatten_study(response.json())

    print(f"{nct}  https://clinicaltrials.gov/study/{nct}")
    for field in ["status", "phases", "enrollment", "sponsor", "sponsor_class"]:
        result = "OK  " if same(row[field], live[field]) else "DIFF"
        print(f"  {result} {field}: mine={row[field]} | live={live[field]}")

    live_year = (live["start_date"] or "")[:4]
    result = "OK  " if same(row["start_year"], live_year) else "DIFF"
    print(f"  {result} start_year: mine={row['start_year']} | live={live_year}")

    # Labels you added can't be checked by the API, so eyeball these
    print(f"  label: {row['phases']} -> {row['phase']}  |  {row['sponsor_class']} -> {row['sponsor_type']}")
    print()
    time.sleep(1)

NCT06330038  https://clinicaltrials.gov/study/NCT06330038
  OK   status: mine=RECRUITING | live=RECRUITING
  OK   phases: mine=PHASE4 | live=PHASE4
  OK   enrollment: mine=5384.0 | live=5384
  OK   sponsor: mine=Samsung Medical Center | live=Samsung Medical Center
  OK   sponsor_class: mine=OTHER | live=OTHER
  OK   start_year: mine=2024 | live=2024
  label: PHASE4 -> Phase 4  |  OTHER -> Academic or hospital

NCT03366064  https://clinicaltrials.gov/study/NCT03366064
  OK   status: mine=COMPLETED | live=COMPLETED
  OK   phases: mine=PHASE1 | live=PHASE1
  OK   enrollment: mine=5.0 | live=5
  OK   sponsor: mine=Asan Medical Center | live=Asan Medical Center
  OK   sponsor_class: mine=OTHER | live=OTHER
  OK   start_year: mine=2017 | live=2017
  label: PHASE1 -> Phase 1  |  OTHER -> Academic or hospital

NCT00531076  https://clinicaltrials.gov/study/NCT00531076
  OK   status: mine=TERMINATED | live=TERMINATED
  OK   phases: mine=PHASE1 | live=PHASE1
  OK   enrollment: mine=6.0 | live=6
 